# Séance 5 · Analyser et raconter une histoire avec les données · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

Des données propres, c'est bien. Mais un tableau de 800 lignes ne dit rien à personne. Aujourd'hui tu apprends à **poser les bonnes questions** aux données, à repérer une tendance, une anomalie, une corrélation, à ne pas te faire piéger (corrélation n'est pas causalité, graphiques menteurs, données biaisées), puis à construire un **tableau de bord interactif** avec Plotly.

Ce notebook tourne dans **Google Colab** (rien à installer). Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter.

**Livrable de la séance** : un mini-dashboard que tu présentes à voix haute en 2 minutes, comme devant un client.


## Préparation

Deux jeux de données : **Tips** (244 additions d'un restaurant : montant, pourboire, jour, nombre de convives) et les **Pokémon** de la séance 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

URL_TIPS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    tips = pd.read_csv(URL_TIPS)
    pokemon = pd.read_csv(URL_POKEMON)
    print("Tips :", tips.shape, "· Pokémon :", pokemon.shape)
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger les fichiers :", erreur)
tips.head()

## 1. L'analyse exploratoire : quelles questions poser ?

L'**analyse exploratoire** (EDA en anglais), c'est faire connaissance avec les données avant de conclure quoi que ce soit. Comme un détective qui arrive sur une scène : on regarde tout, on ne juge rien, on note ce qui est bizarre.

Une bonne question d'analyse se reconnaît à trois choses : elle est **précise** (« les pourboires sont-ils plus gros le week-end ? » plutôt que « c'est quoi les pourboires ? »), on peut y **répondre avec les colonnes qu'on a**, et la réponse **intéresse quelqu'un** (le patron du restaurant, ici).

Le réflexe de base : `describe()` pour les nombres, `value_counts()` pour les catégories.

In [ ]:
print(tips.describe().round(2))
print()
print(tips["day"].value_counts())

**Exercice** : écris 3 questions précises sur le dataset Tips (dans un commentaire), puis réponds à la première avec `groupby`. Exemples de départ : le pourboire moyen change-t-il selon le jour ? Les fumeurs laissent-ils plus ? Les grandes tables donnent-elles plus par personne ?

<details><summary>Solution</summary>

```python
# Q1 : le pourboire moyen change-t-il selon le jour ?
# Q2 : les fumeurs laissent-ils un pourboire plus gros ?
# Q3 : le pourboire par personne baisse-t-il avec la taille de la table ?
print(tips.groupby("day")["tip"].mean().round(2))
print(tips.groupby("smoker")["tip"].mean().round(2))
tips["tip_par_personne"] = tips["tip"] / tips["size"]
print(tips.groupby("size")["tip_par_personne"].mean().round(2))
```
</details>

In [ ]:
# À toi

## 2. Repérer une tendance

Une **tendance**, c'est une direction générale : « plus l'addition est grosse, plus le pourboire est gros ». Le graphique roi pour ça : le **nuage de points** (*scatter*). Seaborn ajoute en une ligne la droite qui résume la tendance.

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.regplot(data=tips, x="total_bill", y="tip", scatter_kws={"alpha": 0.5})
plt.xlabel("Addition ($)"); plt.ylabel("Pourboire ($)")
plt.title("Plus l'addition est grosse, plus le pourboire monte")
plt.show()

In [ ]:
# Une tendance par catégorie : le pourboire moyen selon le jour et le moment
tendance = tips.groupby(["day", "time"])["tip"].mean().round(2).unstack()
print(tendance)
tendance.plot(kind="bar", figsize=(7, 4), title="Pourboire moyen par jour et moment")
plt.ylabel("Pourboire ($)"); plt.show()

**Exercice** : sur les Pokémon, y a-t-il une tendance entre `Attack` et `Sp. Atk` ? Trace un `regplot`. Puis regarde la tendance du `Total` moyen par `Generation` avec `groupby` et un graphique en barres.

<details><summary>Solution</summary>

```python
plt.figure(figsize=(7, 4.5))
sns.regplot(data=pokemon, x="Attack", y="Sp. Atk", scatter_kws={"alpha": 0.4})
plt.show()
pokemon.groupby("Generation")["Total"].mean().plot(kind="bar", title="Puissance moyenne par génération")
plt.show()
```
</details>

In [ ]:
# À toi

## 3. Repérer une anomalie

Une **anomalie** (ou *outlier*), c'est une valeur qui sort du lot : un pourboire de 10 $ sur une addition de 7 $, un Pokémon avec 255 points de vie. Ce n'est pas forcément une erreur : parfois c'est **la** chose intéressante à raconter.

L'outil visuel : la **boîte à moustaches** (*boxplot*). La boîte contient la moitié centrale des valeurs, les points isolés au-delà des moustaches sont les suspects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=tips, x="day", y="tip", ax=axes[0])
axes[0].set_title("Pourboires par jour : les points isolés sont les anomalies")
sns.boxplot(data=pokemon, y="HP", ax=axes[1])
axes[1].set_title("Points de vie des Pokémon")
plt.show()

In [ ]:
# La règle classique : anomalie = au-delà de 1,5 fois l'écart entre le 1er et le 3e quart
q1, q3 = pokemon["HP"].quantile([0.25, 0.75])
limite_haute = q3 + 1.5 * (q3 - q1)
anomalies = pokemon[pokemon["HP"] > limite_haute]
print("Limite :", limite_haute, "PV ·", len(anomalies), "Pokémon au-dessus")
anomalies[["Name", "Type 1", "HP", "Total"]].sort_values("HP", ascending=False).head()

**Exercice** : trouve les anomalies de pourboire dans Tips avec la même règle sur la colonne `tip`. Qui a laissé le plus gros pourboire, et sur quelle addition ? Calcule aussi le pourcentage (`tip / total_bill`) : l'anomalie est-elle la même ?

<details><summary>Solution</summary>

```python
q1, q3 = tips["tip"].quantile([0.25, 0.75])
limite = q3 + 1.5 * (q3 - q1)
print(tips[tips["tip"] > limite].sort_values("tip", ascending=False))
tips["pourcentage"] = (tips["tip"] / tips["total_bill"] * 100).round(1)
print(tips.sort_values("pourcentage", ascending=False).head(3))   # un pourboire de 71 % sur une addition de 7 $ !
```
</details>

In [ ]:
# À toi

## 4. Repérer une corrélation

Une **corrélation** mesure à quel point deux colonnes bougent ensemble, entre **-1** (quand l'une monte, l'autre descend toujours) et **+1** (elles montent toujours ensemble) ; **0** = aucun lien. `df.corr()` calcule toutes les paires d'un coup, et une **heatmap** (carte de chaleur) les colorie pour lire d'un coup d'œil.

In [ ]:
colonnes_num = ["total_bill", "tip", "size"]
print(tips[colonnes_num].corr().round(2))

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
plt.figure(figsize=(7, 5.5))
sns.heatmap(pokemon[stats].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Corrélations entre les stats des Pokémon")
plt.show()

**Exercice** : quelles sont les deux stats les plus corrélées ? Et les deux les moins corrélées ? Vérifie avec un nuage de points pour chacune. (Indice : `Speed` et `Defense`, ça s'explique bien avec des Pokémon : un tank lent ou un rapide fragile ?)

<details><summary>Solution</summary>

```python
c = pokemon[stats].corr()
paires = c.where(np.triu(np.ones(c.shape), k=1).astype(bool)).stack().sort_values()
print("Moins corrélées :", paires.index[0], round(paires.iloc[0], 2))
print("Plus corrélées :", paires.index[-1], round(paires.iloc[-1], 2))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pokemon.plot.scatter(x=paires.index[-1][0], y=paires.index[-1][1], alpha=0.4, ax=axes[0])
pokemon.plot.scatter(x=paires.index[0][0], y=paires.index[0][1], alpha=0.4, ax=axes[1])
plt.show()
```
</details>

In [ ]:
# À toi

## 5. Ne pas se faire piéger

### 5a. Corrélation n'est pas causalité

En été, les ventes de glaces montent... et les noyades aussi. Les glaces provoquent-elles les noyades ? Non : la **chaleur** fait les deux. On appelle ça une **variable cachée** (ou *facteur de confusion*). Une corrélation dit « ça bouge ensemble », jamais « l'un cause l'autre ».

In [ ]:
# On fabrique des données où la température cause TOUT, et où glaces et noyades ne se parlent jamais
rng = np.random.default_rng(0)
temperature = rng.uniform(5, 35, 120)                                  # 120 journées, de 5 à 35 °C
glaces = 20 + 8 * temperature + rng.normal(0, 25, 120)                 # ventes de glaces
noyades = 0.5 + 0.12 * temperature + rng.normal(0, 0.8, 120)           # noyades (fictives !)
ete = pd.DataFrame({"temperature": temperature.round(1), "glaces": glaces.round(0), "noyades": noyades.clip(0).round(1)})

print("Corrélation glaces / noyades :", round(ete["glaces"].corr(ete["noyades"]), 2), "← impressionnant, et pourtant...")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(ete["glaces"], ete["noyades"], alpha=0.6); axes[0].set_xlabel("Glaces vendues"); axes[0].set_ylabel("Noyades")
axes[0].set_title("Les glaces causent les noyades ?!")
sc = axes[1].scatter(ete["glaces"], ete["noyades"], c=ete["temperature"], cmap="coolwarm", alpha=0.8)
axes[1].set_title("Non : la couleur (température) explique tout"); plt.colorbar(sc, label="°C")
plt.show()

### 5b. Les graphiques trompeurs

Le même chiffre peut avoir l'air d'une catastrophe ou d'un détail selon la façon de le dessiner. Le piège numéro 1 : **l'axe vertical tronqué** (qui ne part pas de zéro). Regarde les deux graphiques ci-dessous : mêmes données.

In [ ]:
mois = ["Jan", "Fév", "Mar", "Avr", "Mai", "Juin"]
abonnes = [1020, 1035, 1028, 1050, 1062, 1071]   # abonnés d'une chaîne, +5 % en 6 mois

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(mois, abonnes, color="tab:red"); axes[0].set_ylim(1000, 1080)
axes[0].set_title("TROMPEUR : « la chaîne explose ! » (axe qui part de 1000)")
axes[1].bar(mois, abonnes, color="tab:blue"); axes[1].set_ylim(0, 1200)
axes[1].set_title("HONNÊTE : +5 % en 6 mois (axe qui part de 0)")
for ax in axes: ax.set_ylabel("Abonnés")
plt.show()

### 5c. Les biais dans les données

Un **biais**, c'est quand les données ne représentent pas la réalité qu'on croit mesurer. Le plus courant : l'**échantillon non représentatif**. Exemple : pour savoir combien de personnes jouent aux jeux vidéo, tu fais ton sondage... à la sortie d'un magasin de jeux vidéo. Ton chiffre sera énorme, et faux.

In [ ]:
# 1000 personnes : 40 % jouent vraiment. On simule deux façons de sonder.
rng = np.random.default_rng(1)
joue = rng.random(1000) < 0.40
sortie_magasin = rng.random(1000) < np.where(joue, 0.8, 0.05)   # un joueur a 80 % de chance d'y être, un non-joueur 5 %

echantillon_hasard = rng.choice(joue, 100, replace=False)
echantillon_magasin = joue[sortie_magasin][:100]
print("Vraie proportion de joueurs      :", joue.mean() * 100, "%")
print("Sondage au hasard (100 personnes)     :", echantillon_hasard.mean() * 100, "%")
print("Sondage devant le magasin (100)  :", echantillon_magasin.mean().round(2) * 100, "%  ← biaisé !")

**Exercice** : pour chaque affirmation, dis dans un commentaire quel piège se cache (causalité ? graphique ? biais ?), puis vérifie la troisième avec le code.

1. « Les élèves qui ont un ordinateur portable ont de meilleures notes : donnons un portable à tout le monde ! »
2. « Notre appli est passée de 4,1 à 4,3 étoiles : le graphique montre une barre deux fois plus haute. »
3. « 95 % des avis sur notre jeu sont positifs » (seuls les joueurs qui ont fini le jeu ont été invités à noter).

<details><summary>Solution</summary>

```python
# 1. Causalité : variable cachée probable (revenu du foyer, cadre de travail...)
# 2. Graphique trompeur : axe tronqué, 4,1 → 4,3 c'est +5 %, pas +100 %
# 3. Biais d'échantillon : ceux qui ont abandonné le jeu (probablement mécontents) ne sont pas interrogés
rng = np.random.default_rng(2)
a_fini = rng.random(1000) < 0.5
content = rng.random(1000) < np.where(a_fini, 0.95, 0.30)
print("Avis positifs parmi ceux qui ont fini :", content[a_fini].mean().round(2) * 100, "%")
print("Avis positifs chez tous les joueurs   :", content.mean().round(2) * 100, "%")
```
</details>

In [ ]:
# À toi

## 6. Dashboarding : un tableau de bord interactif avec Plotly

Un **tableau de bord** (*dashboard*), c'est une page qui rassemble 3 ou 4 graphiques qui répondent aux questions d'un « client ». Avec **Plotly**, les graphiques sont interactifs : survole un point, zoome, clique sur la légende pour masquer une catégorie. `plotly.express` (abrégé `px`) fait un graphique en une ligne.

In [ ]:
fig = px.scatter(tips, x="total_bill", y="tip", color="day", size="size", hover_data=["time", "smoker"],
                 title="Pourboire selon l'addition (survole les points !)", labels={"total_bill": "Addition ($)", "tip": "Pourboire ($)"})
fig.show()

In [ ]:
# facet_col : un sous-graphique par catégorie, en une seule ligne
fig = px.box(tips, x="day", y="tip", color="smoker", facet_col="time", title="Pourboires par jour, fumeurs ou non, midi et soir")
fig.show()

In [ ]:
# animation_frame : un graphique qui évolue (ici, génération par génération). Appuie sur ▶
fig = px.scatter(pokemon.sort_values("Generation"), x="Attack", y="Defense", color="Type 1", hover_name="Name",
                 animation_frame="Generation", range_x=[0, 200], range_y=[0, 250],
                 title="Attaque vs Défense, génération par génération")
fig.show()

Pour assembler plusieurs graphiques en un seul tableau de bord, on utilise `make_subplots`. Et pour un vrai menu déroulant, `updatemenus` : chaque option dit quelles courbes afficher.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

moyennes = pokemon.groupby("Type 1")[["HP", "Attack", "Defense", "Speed"]].mean().round(1)
dash = make_subplots(rows=1, cols=3, subplot_titles=("Stat moyenne par type", "Attaque vs Défense", "Nombre par génération"))
# graphique 1 : une barre par stat, une seule visible à la fois (le menu choisit)
for i, stat in enumerate(moyennes.columns):
    dash.add_trace(go.Bar(x=moyennes.index, y=moyennes[stat], name=stat, visible=(i == 0)), row=1, col=1)
dash.add_trace(go.Scatter(x=pokemon["Attack"], y=pokemon["Defense"], mode="markers", text=pokemon["Name"],
                          marker=dict(opacity=0.4), name="Pokémon"), row=1, col=2)
generations = pokemon["Generation"].value_counts().sort_index()
dash.add_trace(go.Bar(x=generations.index, y=generations.values, name="Par génération"), row=1, col=3)

boutons = [dict(label=stat, method="update",
                args=[{"visible": [j == i for j in range(4)] + [True, True]}]) for i, stat in enumerate(moyennes.columns)]
dash.update_layout(updatemenus=[dict(buttons=boutons, x=0.0, y=1.25)], showlegend=False, height=420,
                   title="Mini-dashboard Pokémon : choisis la stat dans le menu")
dash.show()

**Exercice** : ajoute un 4e graphique au dashboard (par exemple un histogramme de `Total`, `go.Histogram`) en passant à `cols=4`. N'oublie pas d'ajouter un `True` dans la liste `visible` de chaque bouton.

<details><summary>Solution</summary>

```python
dash = make_subplots(rows=1, cols=4, subplot_titles=("Stat moyenne", "Att. vs Déf.", "Par génération", "Total"))
for i, stat in enumerate(moyennes.columns):
    dash.add_trace(go.Bar(x=moyennes.index, y=moyennes[stat], name=stat, visible=(i == 0)), row=1, col=1)
dash.add_trace(go.Scatter(x=pokemon["Attack"], y=pokemon["Defense"], mode="markers", marker=dict(opacity=0.4)), row=1, col=2)
dash.add_trace(go.Bar(x=generations.index, y=generations.values), row=1, col=3)
dash.add_trace(go.Histogram(x=pokemon["Total"]), row=1, col=4)
boutons = [dict(label=s, method="update", args=[{"visible": [j == i for j in range(4)] + [True, True, True]}])
           for i, s in enumerate(moyennes.columns)]
dash.update_layout(updatemenus=[dict(buttons=boutons, x=0, y=1.25)], showlegend=False, height=400)
dash.show()
```
</details>

In [ ]:
# À toi

### Et Streamlit ?

**Streamlit** transforme un script Python en vraie page web avec des boutons et des menus, sans écrire de HTML. Ça ne tourne pas dans Colab directement (il faut un serveur), mais sur ton ordinateur c'est 3 lignes : `pip install streamlit`, un fichier `app.py`, puis `streamlit run app.py`. Voici à quoi ressemblerait notre dashboard Pokémon (à copier dans `app.py`, pas à exécuter ici) :

```python
import streamlit as st
import pandas as pd
import plotly.express as px

st.title("Mon dashboard Pokémon")
pokemon = pd.read_csv("pokemon.csv")
type_choisi = st.selectbox("Choisis un type", sorted(pokemon["Type 1"].unique()))   # un menu déroulant
stat = st.radio("Stat à afficher", ["HP", "Attack", "Defense", "Speed"])            # des boutons

sous_ensemble = pokemon[pokemon["Type 1"] == type_choisi]
st.metric("Nombre de Pokémon", len(sous_ensemble))                                    # un gros chiffre
st.plotly_chart(px.histogram(sous_ensemble, x=stat, title=f"{stat} des Pokémon {type_choisi}"))
st.dataframe(sous_ensemble.sort_values(stat, ascending=False).head(10))               # un tableau
```

À chaque clic dans le menu, Streamlit relance le script et met la page à jour. C'est l'outil des data scientists pour montrer un résultat à un client sans lui envoyer un notebook.

## 7. Projet (80 min) : ton mini-dashboard et ta présentation « client »

Tu es data scientist, et ton client est soit le **patron du restaurant** (dataset Tips), soit le **créateur d'un jeu Pokémon** (dataset Pokémon), soit quelqu'un d'autre si tu as ton propre dataset de la séance 2. Il n'a pas le temps : il veut 3 graphiques et 2 minutes d'explication.

Consignes :
1. Choisis ton client et écris **3 questions précises** qui l'intéressent (section 1).
2. Réponds à chacune par un graphique Plotly (`px.scatter`, `px.bar`, `px.box`, `px.histogram`...), avec un titre qui donne la réponse (pas « Pourboire vs addition » mais « Le pourboire suit l'addition, sauf le vendredi »).
3. Assemble-les en un dashboard (`make_subplots`, ou 3 `px` avec `facet_col` / `animation_frame`).
4. Vérifie tes pièges : axe qui part de zéro ? corrélation présentée comme une cause ? échantillon représentatif ?
5. Prépare ton pitch de 2 minutes avec le gabarit **constat → preuve → recommandation**.

In [ ]:
# Question 1 :

In [ ]:
# Question 2 :

In [ ]:
# Question 3 :

In [ ]:
# Mon dashboard (3 graphiques assemblés)

### Le gabarit de présentation (2 minutes, comme devant un client)

Remplis les trois cases ci-dessous. Une seule idée par graphique, des chiffres concrets, et une recommandation que le client peut appliquer demain.

| Étape | Ce que tu dis | Exemple |
|---|---|---|
| **Constat** (20 s) | Ce que tu as trouvé, en une phrase | « Les pourboires du dimanche soir sont 20 % plus élevés que la moyenne » |
| **Preuve** (60 s) | Le graphique qui le montre, et comment le lire | « Sur ce graphique, chaque point est une addition... » |
| **Recommandation** (40 s) | Ce que le client devrait faire | « Mettre les meilleurs serveurs le dimanche soir » |

In [ ]:
pitch = {
    "client": "...",
    "constat": "...",
    "preuve": "... (quel graphique, quel chiffre)",
    "recommandation": "...",
    "piege_verifie": "... (axe à zéro ? causalité ? biais ?)",
}
for etape, texte in pitch.items():
    print(f"{etape.upper():16s} {texte}")

## À retenir

- L'**analyse exploratoire** commence par des questions précises, puis `describe()`, `value_counts()`, `groupby()`
- **Tendance** → nuage de points et droite (`regplot`) ; **anomalie** → boîte à moustaches et règle des 1,5 écarts ; **corrélation** → `corr()` et heatmap
- **Corrélation n'est pas causalité** : cherche la variable cachée (la chaleur derrière les glaces et les noyades)
- Un graphique peut mentir : **l'axe vertical doit partir de zéro** pour des barres
- Un **échantillon biaisé** donne un chiffre précis... et faux
- **Plotly** rend les graphiques interactifs ; `make_subplots` + `updatemenus` = un dashboard ; **Streamlit** en fait une page web
- Une présentation client tient en trois temps : **constat, preuve, recommandation**

## Pour montrer aux autres

Pendant les 20 dernières minutes, chacun présente son dashboard en 2 minutes chrono, les autres jouent le client. Trois questions guides :

1. Quel est ton constat principal, en une phrase avec un chiffre ?
2. Quel graphique le prouve, et pourquoi as-tu choisi celui-là ?
3. Quel piège as-tu vérifié avant de conclure (causalité, axe, biais) ?

## Liens gratuits

- Galerie Plotly Express : https://plotly.com/python/plotly-express/
- Galerie Seaborn : https://seaborn.pydata.org/examples/index.html
- Corrélations absurdes (pour rire, et réfléchir) : https://www.tylervigen.com/spurious-correlations
- Streamlit, démarrer en 5 minutes : https://docs.streamlit.io/get-started